# Merging & De-Duplication

In [ ]:
import pandas as pd

# Load both datasets
df1 = pd.read_csv("german_news_raw.csv")
df2 = pd.read_csv("german_news_raw2.csv")

# Concatenate the dataframes
merged_df = pd.concat([df1, df2], ignore_index=True)


# Remove duplicates based on the 'url' column
merged_df = merged_df.drop_duplicates(subset=["url"], keep="first").reset_index(drop=True)

# Show basic info
print(f"Combined shape: {merged_df.shape}")
print(f"Columns: {merged_df.columns.tolist()}")
print(f"Number of duplicates removed: {(df1.shape[0] + df2.shape[0]) - merged_df.shape[0]}")

merged_df.to_csv("german_news_raw_merged.csv", index=False)
print("Merged German news data saved as german_news_raw_merged.csv")




FileNotFoundError: [Errno 2] No such file or directory: 'german_news_raw.csv'

In [3]:
print(merged_df.shape)

(122208, 5)


# Initial Identification & Cleanup

In [5]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
import warnings
import numpy as np

warnings.filterwarnings('ignore')

print("BERTopic libraries imported successfully!")

/Users/nicolas/Desktop/Repos/zhaw_mle/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BERTopic libraries imported successfully!


In [6]:
# Load the merged dataset
df = pd.read_csv('german_news_raw_merged.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['publishedAt'].min()} to {df['publishedAt'].max()}")

# Prepare text data for topic modeling
# Combine title and description, remove missing values
df_topic = df.copy()
df_topic['combined_text'] = df_topic.apply(
    lambda row: f"{row['title']} {row['description']}" 
    if pd.notna(row['title']) and pd.notna(row['description']) 
    else (row['title'] if pd.notna(row['title']) else str(row['description']) if pd.notna(row['description']) else ''),
    axis=1
)

# Remove empty strings and very short texts
df_topic = df_topic[df_topic['combined_text'].str.strip() != ''].copy()
df_topic = df_topic[df_topic['combined_text'].str.len() > 20].copy()  # Minimum length

print(f"\nArticles prepared for topic modeling: {len(df_topic)}")
print(f"Average document length: {np.mean([len(doc) for doc in df_topic['combined_text']]):.1f} characters")
print(f"\nSample text (first 200 chars):")
print(df_topic['combined_text'].iloc[0][:200] if len(df_topic) > 0 else "No data")


Dataset shape: (122208, 5)
Columns: ['publishedAt', 'title', 'source', 'description', 'url']

Date range: 2020-11-01 02:54:57+00:00 to 2025-10-31 16:32:16+00:00

Articles prepared for topic modeling: 122208
Average document length: 237.0 characters

Sample text (first 200 chars):
Formel 1: Darauf muss man beim Großen Preis der Emilia Romagna achten


In [7]:
# Extract documents for topic modeling
documents = df_topic['combined_text'].tolist()

print(f"Total documents: {len(documents)}")

# Initialize BERTopic with optimized settings for news articles
print("\nInitializing BERTopic model...")
print("This may take a few minutes...")

# Use a lightweight embedding model for faster processing
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Configure dimensionality reduction and clustering
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Initialize BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=10,
    verbose=True,
    language='german'
)

# Fit the model
print("\nFitting the model to documents...")
topics, probs = topic_model.fit_transform(documents)

print(f"\nModel training completed!")
print(f"Number of topics discovered: {len(set(topics)) - (1 if -1 in topics else 0)}")
if -1 in topics:
    print(f"Outliers (topic -1): {topics.count(-1)} documents")


Total documents: 122208

Initializing BERTopic model...
This may take a few minutes...


2025-10-31 18:10:14,691 - BERTopic - Embedding - Transforming documents to embeddings.



Fitting the model to documents...


Batches: 100%|██████████| 3819/3819 [03:42<00:00, 17.17it/s]
2025-10-31 18:13:58,280 - BERTopic - Embedding - Completed ✓
2025-10-31 18:13:58,281 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-31 18:15:30,014 - BERTopic - Dimensionality - Completed ✓
2025-10-31 18:15:30,018 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PAR


Model training completed!
Number of topics discovered: 266
Outliers (topic -1): 48056 documents


In [8]:
# Get topic information and display TOP 50 TOPICS
topic_info = topic_model.get_topic_info()

print("=" * 80)
print("TOP 50 TOPICS WITH KEYWORDS")
print("=" * 80)

# Filter out outlier topic (-1) for display
topics_to_show = topic_info[topic_info['Topic'] != -1].head(50)

print(f"\nTotal topics discovered (excluding outliers): {len(topic_info[topic_info['Topic'] != -1])}")
print(f"Documents assigned to topics: {topics_to_show['Count'].sum()}")
if -1 in topics:
    outliers_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if len(topic_info[topic_info['Topic'] == -1]) > 0 else 0
    print(f"Outliers (unassigned): {outliers_count}")

print("\n" + "=" * 80)
print(f"{'Rank':<6} {'Topic ID':<10} {'Articles':<12} {'Topic Name':<40} {'Top Keywords'}")
print("=" * 80)

for idx, (_, row) in enumerate(topics_to_show.iterrows(), 1):
    topic_id = row['Topic']
    count = row['Count']
    name = row['Name']
    
    # Get top words for this topic
    top_words = topic_model.get_topic(topic_id)
    words_str = ", ".join([word for word, score in top_words[:10]])
    
    # Shorten topic name if too long
    display_name = name[:37] + "..." if len(name) > 40 else name
    
    print(f"{idx:<6} {topic_id:<10} {count:<12} {display_name:<40} {words_str}")


TOP 50 TOPICS WITH KEYWORDS

Total topics discovered (excluding outliers): 266
Documents assigned to topics: 42541
Outliers (unassigned): 48056

Rank   Topic ID   Articles     Topic Name                               Top Keywords
1      0          4619         0_corona_impfstoff_coronavirus_pandemie  corona, impfstoff, coronavirus, pandemie, impfung, biontech, astrazeneca, impfungen, inzidenz, covid
2      1          2519         1_klimaschutz_klima_co2_klimakonferenz   klimaschutz, klima, co2, klimakonferenz, weltklimakonferenz, klimakrise, lützerath, klimawandel, emissionen, aktivisten
3      2          1930         2_stream_nord_gas_russland               stream, nord, gas, russland, öl, pipeline, russisches, gazprom, ostsee, erdgas
4      3          1638         3_kriminalität_polizei_mann_prozess      kriminalität, polizei, mann, prozess, täter, jähriger, kokain, drogen, wurde, messer
5      4          1604         4_spd_scholz_fdp_olaf                    spd, scholz, fdp, olaf, p

In [9]:
df_to_filter = df.copy()

# first set of words to filter out from headlines (case-insensitive)
final_words_to_filter = [
    'täter', 'kokain', 'drogen', 'messer', 'schulen', 'eltern', 'bildung', 'lehrer', 'mutter', 
    'formel', 'verstappen', 'hamilton', 'rennen', 'weihnachtsmarkt', 'weihnachten', 'glühwein', 
    'tourismus', 'urlaub', 'hotels', 'mali', 'sudan', 'niger', 'kongo'
    
]

print("First round of filtering...")
print(f"Current dataset size (from german_news_raw_merged): {len(df_to_filter)} articles")

# Create a mask to identify articles with irrelevant headlines
# Check if title contains any of the final filter words (case-insensitive)
filter_mask_v1 = df_to_filter['title'].str.lower().str.contains('|'.join(final_words_to_filter), na=False, regex=True)

# Count articles to be removed
articles_to_remove_v1 = filter_mask_v1.sum()
print(f"Articles to remove in this round: {articles_to_remove_v1} ({articles_to_remove_v1/len(df_to_filter)*100:.2f}%)")

# Filter the dataframe (keep articles that DON'T match the filter)
df_filtered_v1 = df_to_filter[~filter_mask_v1].copy()

print(f"Filtered dataset size: {len(df_filtered_v1)} articles")
print(f"Total articles removed in this round: {len(df_to_filter) - len(df_filtered_v1)}")
print(f"Total articles removed from original: {len(df) - len(df_filtered_v1)} ({(len(df) - len(df_filtered_v1))/len(df)*100:.2f}%)")

# Save the final filtered dataset
output_filename = 'german_news_v1.csv'

# Save to CSV (already has the correct columns)
df_filtered_v1.to_csv(output_filename, index=False)


First round of filtering...
Current dataset size (from german_news_raw_merged): 122208 articles
Articles to remove in this round: 5070 (4.15%)
Filtered dataset size: 117138 articles
Total articles removed in this round: 5070
Total articles removed from original: 5070 (4.15%)


# Second Round of Identification & Cleanup

In [10]:
# Load the merged dataset
df = pd.read_csv('german_news_v1.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nDate range: {df['publishedAt'].min()} to {df['publishedAt'].max()}")

# Prepare text data for topic modeling
# Combine title and description, remove missing values
df_topic = df.copy()
df_topic['combined_text'] = df_topic.apply(
    lambda row: f"{row['title']} {row['description']}" 
    if pd.notna(row['title']) and pd.notna(row['description']) 
    else (row['title'] if pd.notna(row['title']) else str(row['description']) if pd.notna(row['description']) else ''),
    axis=1
)

# Remove empty strings and very short texts
df_topic = df_topic[df_topic['combined_text'].str.strip() != ''].copy()
df_topic = df_topic[df_topic['combined_text'].str.len() > 20].copy()  # Minimum length

print(f"\nArticles prepared for topic modeling: {len(df_topic)}")
print(f"Average document length: {np.mean([len(doc) for doc in df_topic['combined_text']]):.1f} characters")
print(f"\nSample text (first 200 chars):")
print(df_topic['combined_text'].iloc[0][:200] if len(df_topic) > 0 else "No data")


Dataset shape: (117138, 5)
Columns: ['publishedAt', 'title', 'source', 'description', 'url']

Date range: 2020-11-01 02:54:57+00:00 to 2025-10-31 16:32:16+00:00

Articles prepared for topic modeling: 117138
Average document length: 236.9 characters

Sample text (first 200 chars):
Pandemische Echtzeitradikalisierung der Corona-Leugner - Podcast Die Bewegung der Corona-Leugner ist nicht mehr mit der aus dem Sommer vergleichbar, sie hat sich blitzradikalisiert, schrieb Sascha Lob


In [11]:
# Extract documents for topic modeling
documents = df_topic['combined_text'].tolist()

print(f"Total documents: {len(documents)}")

# Initialize BERTopic with optimized settings for news articles
print("\nInitializing BERTopic model...")
print("This may take a few minutes...")

# Use a lightweight embedding model for faster processing
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Configure dimensionality reduction and clustering
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Initialize BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    top_n_words=10,
    verbose=True,
    language='german'
)

# Fit the model
print("\nFitting the model to documents...")
topics, probs = topic_model.fit_transform(documents)

print(f"\nModel training completed!")
print(f"Number of topics discovered: {len(set(topics)) - (1 if -1 in topics else 0)}")
if -1 in topics:
    print(f"Outliers (topic -1): {topics.count(-1)} documents")


Total documents: 117138

Initializing BERTopic model...
This may take a few minutes...


2025-10-31 18:26:45,476 - BERTopic - Embedding - Transforming documents to embeddings.



Fitting the model to documents...


Batches: 100%|██████████| 3661/3661 [03:29<00:00, 17.45it/s]
2025-10-31 18:30:16,446 - BERTopic - Embedding - Completed ✓
2025-10-31 18:30:16,447 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-31 18:31:35,246 - BERTopic - Dimensionality - Completed ✓
2025-10-31 18:31:35,250 - BERTopic - Cluster - Start clustering the reduced embeddings
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PAR


Model training completed!
Number of topics discovered: 249
Outliers (topic -1): 46417 documents


In [12]:
# Get topic information and display TOP 50 TOPICS
topic_info = topic_model.get_topic_info()

print("=" * 80)
print("TOP 50 TOPICS WITH KEYWORDS")
print("=" * 80)

# Filter out outlier topic (-1) for display
topics_to_show = topic_info[topic_info['Topic'] != -1].head(50)

print(f"\nTotal topics discovered (excluding outliers): {len(topic_info[topic_info['Topic'] != -1])}")
print(f"Documents assigned to topics: {topics_to_show['Count'].sum()}")
if -1 in topics:
    outliers_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if len(topic_info[topic_info['Topic'] == -1]) > 0 else 0
    print(f"Outliers (unassigned): {outliers_count}")

print("\n" + "=" * 80)
print(f"{'Rank':<6} {'Topic ID':<10} {'Articles':<12} {'Topic Name':<40} {'Top Keywords'}")
print("=" * 80)

for idx, (_, row) in enumerate(topics_to_show.iterrows(), 1):
    topic_id = row['Topic']
    count = row['Count']
    name = row['Name']
    
    # Get top words for this topic
    top_words = topic_model.get_topic(topic_id)
    words_str = ", ".join([word for word, score in top_words[:10]])
    
    # Shorten topic name if too long
    display_name = name[:37] + "..." if len(name) > 40 else name
    
    print(f"{idx:<6} {topic_id:<10} {count:<12} {display_name:<40} {words_str}")


TOP 50 TOPICS WITH KEYWORDS

Total topics discovered (excluding outliers): 249
Documents assigned to topics: 42381
Outliers (unassigned): 46417

Rank   Topic ID   Articles     Topic Name                               Top Keywords
1      0          4512         0_corona_impfstoff_coronavirus_pandemie  corona, impfstoff, coronavirus, pandemie, impfung, biontech, astrazeneca, impfungen, inzidenz, covid
2      1          4279         1_preis_auszeichnung_auszeichnungen_e... preis, auszeichnung, auszeichnungen, erhält, literatur, ausgezeichnet, sie, lesen, hier, informationen
3      2          3103         2_inflation_ezb_zinsen_prozent           inflation, ezb, zinsen, prozent, zentralbank, leitzins, banken, notenbank, tagesgeld, euroraum
4      3          1873         3_stream_nord_gas_russland               stream, nord, gas, russland, öl, pipeline, russisches, gazprom, ostsee, gaslieferungen
5      4          1362         4_israel_gaza_hamas_nahost               israel, gaza, hamas, nah